# 03 · Build a portable geospatial layer

**Atlas source reading copy · browser-network**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-WorldView/blob/8e8feea094d489192b8a101069d32fa1328c764f/content/03_build_your_own_layer.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability.

**Provenance:** `content/03_build_your_own_layer.ipynb` at `8e8feea094d489192b8a101069d32fa1328c764f`; original SHA-256 `75ea13c745e2ec31cfdbb288730b25dce894819ae0efc2d8b6df462082af0f00`. Source-specific license/notices retained under notices; data and media have separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# 03 · Build a portable geospatial layer

**Mission:** create and validate GeoJSON with units, provenance, and freshness, then import it into both viewers. Allow 30 minutes. Prerequisite: Lab 00.

**Run:** choose the Python (Pyodide) kernel in JupyterLite, then Run → Run All Cells. First use downloads Matplotlib. All training data are **SYNTHETIC** unless explicitly labeled LIVE. Downloads appear below and in `exports/`; the browser filesystem is separate from your computer. Each lab runs independently.

In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install("matplotlib")
import json, math
from pathlib import Path
from IPython.display import display, HTML
import matplotlib.pyplot as plt
from worldview_lab import (load, export, map_layer, chart_style, haversine_km,
    wilson_interval, shortest_times, freshness, normalize_quakes,
    validate_geojson, brief_request, validate_brief)
chart_style()

In [ ]:
sites=load('facilities.geojson')
for feature in sites['features']:
    p=feature['properties']
    p['freshness']=freshness(p['observed_at'],'2026-09-14T12:00:00Z')
assert validate_geojson(sites)
map_layer(sites,value='capacity',title='SYNTHETIC facilities — capacity in exercise service slots')
export('my-facilities.geojson',sites)

## Background tiles and saved browser files

The **OpenStreetMap background** checkbox controls the street map beneath your facilities. If tiles are unavailable, the map keeps your four synthetic points and their popups visible and explains the failure. Check the box again to retry, or call `map_layer(sites, value="capacity", basemap=False)` to work with just the data layer. This still needs the Leaflet library; it does not download offline street maps.

If an older copy shows **Access blocked** tiles, open this lab in a private browser window to load the updated helper and notebook without deleting your saved work. JupyterLite retains browser copies, so refreshing alone may keep the old `worldview_lab.py`. If you replace that file in an existing session, restart the Python kernel before running all cells.

The renderer preserves the browser's real referring origin, as required by the [OpenStreetMap tile usage policy](https://operations.osmfoundation.org/policies/tiles/). It removes a failed background layer instead of repeatedly retrying. Third-party tile services can still be unavailable. The one-time Matplotlib font-cache message is normal initialization, not a map error.

## Coordinate order and provenance

GeoJSON positions use **longitude, latitude**; Leaflet’s direct `LatLng` API uses **latitude, longitude**. `L.geoJSON` performs the conversion. Values in range can still be swapped, so also check a known location and map extent. These points are invented facilities, not a directory.

Download the layer, open **2D EOC → Import GeoJSON** or **3D scenes → Import GeoJSON**, and select it. The import is local to that browser tab. Only Point and Polygon geometries are accepted by this teaching importer; a rejected file reports a reason.

In [ ]:
import copy
bad=copy.deepcopy(sites)
bad['features'][0]['geometry']['coordinates']=[400,95]
try: validate_geojson(bad)
except ValueError as error: print('Expected rejection:',error)
print('A stale observation:',freshness('2026-09-14T08:00:00Z','2026-09-14T12:00:00Z'))

## Lab challenge

Add a fifth fictional facility with a source, UTC observation time, and capacity. Change the scenario clock by two hours and compare freshness. **Checkpoint:** all five points appear in both viewers; bad coordinates are rejected; no source labels disappear.

**Astra prompt:** “Review this GeoJSON schema and validation code. Propose edge cases for coordinate order, null geometry, missing source, and future timestamps. Explain each failure before proposing a fix.”

## Sources and attribution

[RFC 7946](https://www.rfc-editor.org/rfc/rfc7946) defines GeoJSON. [Leaflet reference](https://leafletjs.com/reference.html) documents GeoJSON ingestion.

World View inspiration: [Kevin (Khoa) To, MIT](https://github.com/kevtoe/worldview). Original lab code, figures, and synthetic fixtures: JupyterLite WorldView contributors, MIT. See the [book references](https://jltobias.github.io/JupyterLite-WorldView/book/references.html) and repository notices for dependency and service terms.